# 격자-위험지수 조인 (4개 시·구)

`01._격자_(4개_시·구).geojson`과 `08_격자_종합위험지수.csv`를 `gid` 기준으로 병합합니다.

**출력:** `epdo_analysis/output/15_격자위험도_조인.geojson / .shp`

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

## 경로 설정

노트북이 프로젝트 루트(`LH-Data-Analyze/`)에 위치한다고 가정합니다.

In [ ]:
BASE_DIR = Path(".").resolve()

# 입력 파일
INPUT_GRID = BASE_DIR / "data" / "01._격자_(4개_시·구).geojson"
INPUT_RISK = BASE_DIR / "epdo_analysis" / "output" / "08_격자_종합위험지수.csv"

# 출력 파일
OUT_GEO = BASE_DIR / "epdo_analysis" / "output" / "15_격자위험도_조인.geojson"
OUT_SHP = BASE_DIR / "epdo_analysis" / "output" / "15_격자위험도_조인.shp"

print("BASE_DIR :", BASE_DIR)
print(f"입력 격자  : {'OK' if INPUT_GRID.exists() else 'NOT FOUND'}  {INPUT_GRID}")
print(f"입력 위험도: {'OK' if INPUT_RISK.exists() else 'NOT FOUND'}  {INPUT_RISK}")

## 데이터 로드

In [ ]:
gdf = gpd.read_file(INPUT_GRID)
df  = pd.read_csv(INPUT_RISK, encoding="utf-8-sig", low_memory=False)

print(f"격자 행 수  : {len(gdf):,}  CRS: {gdf.crs}")
print(f"위험지수 행 수: {len(df):,}")
print(f"위험지수 컬럼: {df.columns.tolist()}")

## 조인 및 좌표계 변환

In [ ]:
gdf["gid"]        = gdf["gid"].astype(str)
df["grid_gid"]    = df["grid_gid"].astype(str)

out = gdf.merge(df, left_on="gid", right_on="grid_gid", how="left")
out = out.to_crs("EPSG:4326")

print(f"조인 결과 행 수: {len(out):,}")
out.head()

## 파일 저장

In [ ]:
OUT_GEO.parent.mkdir(parents=True, exist_ok=True)

out.to_file(OUT_GEO, driver="GeoJSON")

# SHP 저장: 컬럼명 10자 제한 대응
def shorten_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    rename_map = {}
    used = set()
    for col in gdf.columns:
        if col == "geometry":
            continue
        if len(col) > 10:
            base = col[:10]
            short = base
            idx = 1
            while short in used:
                short = base[: 10 - len(str(idx))] + str(idx)
                idx += 1
            rename_map[col] = short
            used.add(short)
        else:
            used.add(col)
    if rename_map:
        print("[SHP 컬럼 단축]")
        for orig, short in rename_map.items():
            print(f"  {orig!r:35s} → {short!r}")
    return gdf.rename(columns=rename_map)

out_shp = shorten_columns(out.copy())
out_shp.to_file(OUT_SHP, driver="ESRI Shapefile", encoding="utf-8")

print("saved:", OUT_GEO)
print("saved:", OUT_SHP)